In [2]:
import pandas as pd
import numpy as np
import seaborn as sns

flights = sns.load_dataset("flights")
flights

,year,month,passengers
0,1949,Jan,112
1,1949,Feb,118
2,1949,Mar,132
3,1949,Apr,129
4,1949,May,121
...,...,...,...
139,1960,Aug,606
140,1960,Sep,508
141,1960,Oct,461
142,1960,Nov,390


### 1번문제 
FLIGHTS 데이터  
import seaborn as sns
flights = sns.load_dataset("flights")

연도별 · 월별 평균 승객 수 비교
- 연도(year)별 총 승객 수와 평균 승객 수를 구하라.
- 월(month)별 평균 승객 수를 구하라.
- 어느 연도가 전체적으로 가장 승객이 많았는지
- 어느 달이 평균적으로 가장 바쁜(승객이 많은) 달인지를 각각 찾고,

이 두 결과를 비교해서 “항공 수요가 언제 정점에 가까운지” 해석해보라.

In [3]:
cnt_psg = flights.groupby("year")["passengers"].sum("passengers")
avg_psg = flights.groupby("year")["passengers"].mean().astype(int)

pd.DataFrame([cnt_psg, avg_psg], index = ["연도별 총 승객 수", "연도별 평균 승객 수"])

year,1949,1950,1951,1952,1953,1954,1955,1956,1957,1958,1959,1960
연도별 총 승객 수,1520,1676,2042,2364,2700,2867,3408,3939,4421,4572,5140,5714
연도별 평균 승객 수,126,139,170,197,225,238,284,328,368,381,428,476


In [4]:
avg_month_psg = flights.groupby("month")[["passengers"]].mean().astype(int)
print(type(avg_month_psg))
avg_month_psg

<class 'pandas.core.frame.DataFrame'>


C:\Users\user\AppData\Local\Temp\ipykernel_26420\3692491416.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  avg_month_psg = flights.groupby("month")[["passengers"]].mean().astype(int)


,passengers
month,
Jan,241
Feb,235
Mar,270
Apr,267
May,271
Jun,311
Jul,351
Aug,351
Sep,302


In [5]:
cnt_psg[cnt_psg == cnt_psg.max()].index

Index([1960], dtype='int64', name='year')

In [6]:
# avg_month_psg[avg_month_psg == avg_month_psg.max()]
avg_month_psg[avg_month_psg["passengers"] == avg_month_psg["passengers"].max()].index

CategoricalIndex(['Jul', 'Aug'], categories=['Jan', 'Feb', 'Mar', 'Apr', ..., 'Sep', 'Oct', 'Nov', 'Dec'], ordered=False, dtype='category', name='month')

In [11]:
print("항공수요가 언제 정점에 가까워졌는지?\n==>")
print(f"{cnt_psg[cnt_psg == cnt_psg.max()]} \n {avg_month_psg[avg_month_psg["passengers"] == avg_month_psg["passengers"].max()]}")

항공수요가 언제 정점에 가까워졌는지?
==>
year
1960    5714
Name: passengers, dtype: int64 
        passengers
month            
Jul           351
Aug           351



### 2번문제

승객 수 기반 “트래픽 레벨” 만들기
passengers 값을 기준으로, 다음과 같이 트래픽 수준을 나누어라.
- 승객 수가 상위 20% 이상 → "high-traffic"
- 승객 수가 50% ~ 80% 분위수 → "mid-traffic"
- 나머지 → "low-traffic"

이 기준으로 traffic_level이라는 파생변수를 생성하라. (힌트: quantile() 또는 pd.qcut() + 조건문 활용)
- month(월) × traffic_level 조합별 평균 passengers를 계산하라.
- 평균 승객 수가 가장 많은 조합 TOP 3를 내림차순으로 출력하라.
- TOP 1 조합이 나오는 이유를,
- 해당 월이 포함된 계절(예: 여름 휴가철, 연말 등)
- 전반적인 연도 증가 추세와 연결하여 데이터적으로 해석해보라.

In [42]:
flights["traffic_level"] = pd.qcut(flights.passengers, q = [0, 0.5, 0.8, 1], labels = ["low-traffic", "mid-traffic", "high-traffic"])

#transform을 통해 인덱스 맞추기(원래 행의 개수만큼 방송(broadcast))
flights["월별 트래픽 레벨"] = flights.groupby(["month", "traffic_level"])["passengers"].transform("mean")

top3_flights = flights.passengers.sort_values(ascending=False).head(3)
flights
top3_flights

C:\Users\user\AppData\Local\Temp\ipykernel_26420\1485933590.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  flights["월별 트래픽 레벨"] = flights.groupby(["month", "traffic_level"])["passengers"].transform("mean")


138    622
139    606
127    559
Name: passengers, dtype: int64

### 3번문제 
“안정적 성수기(Stable Peak Season)” 찾기
먼저, month를 이용해 다음과 같이 season 컬럼을 만들어라.
- 12, 1, 2월 → "winter"
- 3, 4, 5월 → "spring"
- 6, 7, 8월 → "summer"
- 9, 10, 11월 → "fall"

연도를 다음 두 그룹으로 나누는 period 변수를 만들어라.
- year <= 1952 → "early"
- year > 1952 → "late"

season × period 조합별로 passengers의 평균(mean), 분산(var)을 모두 구하라.
- 평균 승객 수는 높고, 분산은 낮은 조합을 “안정적 성수기(Stable Peak Season)”라고 정의하고,
- 이 기준에 따라 TOP 3 season-period 조합을 선정하라.
- 선택된 season-period 조합의 공통적인 특징을
  - 계절(여름/겨울/…?)
  
항공 수요의 연도별 증가 추세 관점에서 해석해보라.
